## 11. Optional: Hyperparameter Tuning

Tuning tries multiple parameter combinations with `TimeSeriesSplit`. This can take a long time. Use it only after the simple training run works.


In [ ]:
from pathlib import Path
import os
import sys
import importlib
import pandas as pd
from dotenv import load_dotenv

# Make local imports work when the notebook is opened from another folder.
PIPELINE_DIR = Path.cwd()
if not (PIPELINE_DIR / 'loader.py').exists():
    PIPELINE_DIR = Path.cwd() / 'pipeline'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.append(str(PIPELINE_DIR))

from loader import LoaderStorage
from targets import add_delay_class_target, DELAY_CLASS_ORDER
from split import chronological_train_val_test_split
from features import  check_columns
from evaluate import plotConfusionMatrix
import train
importlib.reload(train)
from train import TrainingConfig, run_training
import models
importlib.reload(models)


In [ ]:
# Change these paths to your actual dataset location.
DATA_ROOT = "s3://data-mining"
INPUT_PATH = 'data/features/feature_engineered.parquet'

# Column names used by the current pipeline.
DELAY_COLUMN = 'ArrDelayMinutes'
TIME_COLUMN = 'CRSDepDateTime_UTC'
TARGET_COLUMN = 'delay_class'

# Use a small sample while learning/debugging. Set to 1.0 for the final run.
SAMPLE_FRAC = 0.2

# Good first choices: 'dummy', 'logistic_regression', 'random_forest', 'hist_gradient_boosting', 'xgboost', 'svc'.
MODEL_NAMES = ['dummy', 'logistic_regression', 'random_forest', 'hist_gradient_boosting', 'xgboost', 'svc']

OUTPUT_DIR = 'outputs/training_notebook'


In [ ]:
# Uncomment this cell when you are ready for a slower tuning run.
for n in MODEL_NAMES:

  tuned_config = TrainingConfig(
      data_root=DATA_ROOT,
      input_path=INPUT_PATH,
      output_dir='outputs/training_notebook_tuned',
      model_name=MODEL_NAME,
      delay_column=DELAY_COLUMN,
      target_column=TARGET_COLUMN,
      time_column=TIME_COLUMN,
      sample_frac=SAMPLE_FRAC,
      weights=None, # ignored during tuning; class weights are sampled from intervals in RandomizedSearchCV.
      tune=True,
      n_iter=10,
      cv_splits=3,
  )
  tuned_metrics,best_params = run_training(tuned_config)
  display(pd.Series(tuned_metrics, name='tuned_pipeline_metrics'))
  if best_params:
      print("Best hyperparameters found during tuning:")
      for param, value in best_params.items():
          print(f"{param}: {value}")